> **Kernel: use this repository's `.venv` (Python 3.11).** In VS Code click **Select Kernel** (top right) →
> **Python Environments…** → **.venv**, the one whose path ends in `hf-visual\.venv\Scripts\python.exe`.
> If it isn't listed, run `uv sync` in the `hf-visual` folder and reload VS Code. Any other Python (including the
> base or a conda environment) fails with `ModuleNotFoundError`. See `examples/README.md` for the full setup.

# Unit explorer

Browse the sorted units of a recording day across all its stimulus runs. Pick a run, filter the units by label and by
how selective they are, and step through them. One Kilosort sorting covers the whole day, so unit numbers are the
same in every run and each unit is shown in **every** run side by side.

Run all cells, then use the controls at the bottom. Loading takes about a minute per run over the network; after
that the explorer is instant.

In [ ]:
from pathlib import Path

import ipywidgets as w
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

from open_rigs_hf_visual import receptive_fields as rf
from open_rigs_hf_visual import spikes as sp
from open_rigs_hf_visual.stimulus_runs import load_run, score_units

## Settings

Each day lists its stimulus runs, one per SpikeGLX gate (`_gN`), with the kind of run:

| kind | stimulus | events | condition |
|---|---|---|---|
| `trials` | drifting gratings, dot motion (flat CSV files with a trial table) | trials | direction / dot velocity, `blank` |
| `reversals` | checkerboard (flat CSV files, no trial table) | every photodiode flip | phase: `rising` / `falling` |
| `sparse_noise` | sparse noise (flat CSV files + `.bin` frame log) | every frame | receptive field maps |
| `textures` | open-rigs texture sessions (`ses-...` folder) | presentations | texture family (`SN_42`, `SImg_605` ...), `blank` |

`kilosort_dir` is the day's CatGT processed folder. Runs without probe data can't be included: on 2026-10-08 the
texture session (`_g2`) is missing from `Processed_1`.

In [ ]:
subject = Path(r"Z:\ibn-vision\DATA\SUBJECTS\M26031")

days = {
    "2026-10-18": dict(
        kilosort_dir=subject / r"ephys\20261018\Processed_0",
        runs={
            "Gratings": dict(kind="trials", protocol="DriftingGratings", gate="g0"),
            "Checkerboard": dict(kind="reversals", protocol="Checkerboard", gate="g1"),
            "SparseNoise": dict(kind="sparse_noise", protocol="SparseNoise", gate="g2"),
            "Textures": dict(kind="textures", gate="g3"),
        },
        folder=subject / r"stimuli\20261018\M26031_20261018_0",
        ephys=subject / r"ephys\20261018\20261018_0",
        run_prefix="M26031_20261018_0",
    ),
    "2026-10-08": dict(
        kilosort_dir=subject / r"ephys\20261008\Processed_1",
        runs={
            "DotMotion": dict(kind="trials", protocol="DotMotion", start=-1, gate="g0"),
            "Gratings": dict(kind="trials", protocol="DriftingGratings", gate="g1"),
        },
        folder=subject / r"stimuli\20261008\M26031_20261008_1",
        ephys=subject / r"ephys\20261008\20261008_1",
        run_prefix="M26031_20261008_1",
    ),
}
day = "2026-10-18"  # which day to explore

## Load every run and score every unit

Per run and unit (`stimulus_runs.score_units`):

- `baseline_rate`, `stim_rate`, `response`: mean firing rate (Hz) before onset, during the stimulus, and the difference;
- `selectivity`: one-way ANOVA F of the stimulus-period rate across conditions (about 1 = no effect). For sparse
  noise it is the receptive field strength instead: the peak z-score of the ON/OFF maps divided by its value with
  the frames shifted out of register with the spikes (again about 1 = no receptive field);
- `selectivity_stim`: the same with blank trials left out, i.e. tuning among stimuli;
- `label` (Kilosort "good" / "mua") and probe position (`x`, `y` = distance from the tip, µm).

In [ ]:
settings = days[day]
runs, scores = {}, {}
for name, spec in settings["runs"].items():
    spec = dict(spec)
    gate = spec.pop("gate")
    if spec["kind"] == "textures":
        spec["root"] = next(settings["folder"].glob("ses-*"))
    else:
        spec["folder"] = settings["folder"]
    run = load_run(name, ephys_root=settings["ephys"] / f"{settings['run_prefix']}_{gate}",
                   kilosort_dir=settings["kilosort_dir"], **spec)
    runs[name], scores[name] = run, score_units(run)
    print(f"{name} ({gate}, {run.kind}): {len(run.events)} events, stimulus {run.stim_window[1]:.2f} s, "
          f"{len(scores[name])} units, {(scores[name]['selectivity'] > 3).sum()} with selectivity > 3")
names = list(runs)

## Overview

Left: every unit's selectivity in every run (log scale), units sorted by their best selectivity, so units driven by
several stimuli or by just one stand out. Right: positions on the probe, coloured by best selectivity.

In [ ]:
table = pd.concat({name: s["selectivity"] for name, s in scores.items()}, axis=1)
best = np.log10(table.max(axis=1).clip(lower=0.1))
order = best.sort_values(ascending=False).index
fig, axes = plt.subplots(1, 2, figsize=(13, 6), gridspec_kw={"width_ratios": [1, 1.2]})
image = axes[0].imshow(np.log10(table.loc[order].clip(lower=0.1)), aspect="auto", interpolation="nearest",
                       cmap="magma", vmin=-0.5, vmax=2.5)
axes[0].set_xticks(range(len(names)), names, rotation=30)
axes[0].set(ylabel="unit (most selective at top)", title="log10 selectivity per run")
fig.colorbar(image, ax=axes[0])
first = scores[names[0]]
image = axes[1].scatter(first["x"], first["y"], c=best.reindex(first.index), s=14, cmap="magma", vmin=-0.5, vmax=2.5)
axes[1].set(xlabel="x on probe (µm)", ylabel="distance from tip (µm)", title="Probe position, best selectivity")
fig.colorbar(image, ax=axes[1])
plt.tight_layout()

## Explorer

- **Recording**: the run used to filter and sort units. **Units**: Kilosort label.
- **Min selectivity**: hide units below this selectivity in that run (log slider).
- **Ignore blank**: filter and sort by tuning among stimuli (`selectivity_stim`).
- **Sort by**, **previous / next**: order of and stepping through the unit list.
- **Show all recordings**: the unit in every run of the day, or only the chosen run.

Each column is a run. Event-based runs: raster and PSTH grouped by condition (shaded = stimulus), then the tuning
curve (mean rate ± SEM in the stimulus period; dashed = baseline). Sparse noise: ON (white) and OFF (black) maps of
the change in firing 30–130 ms after a square appeared (red = more firing; the box marks the strongest square),
then the response time course at that square. The probe map on the right shows all units (grey), units passing the filters
(blue) and the current unit (red star).

In [ ]:
rf_lags = np.arange(-0.04, 0.3, 0.01)


def plot_window(run):
    """Time range to plot around each event, from the spacing of the events."""
    spacing = float(np.median(np.diff(run.events["onset"])))
    return (-min(0.5, spacing / 2), min(1.5, spacing))


def plot_events_column(fig, cell, run, unit, row):
    inner = cell.subgridspec(3, 1, height_ratios=[2.2, 1, 1], hspace=0.35)
    ax_raster = fig.add_subplot(inner[0])
    ax_psth = fig.add_subplot(inner[1], sharex=ax_raster)
    ax_tuning = fig.add_subplot(inner[2])
    times, events = run.spikes.spikes(unit), run.events
    sp.plot_unit(times, events["onset"], events["condition"], window=plot_window(run), bin_size=0.01,
                 smooth=0.01, shade=run.stim_window, axes=(ax_raster, ax_psth))
    ax_raster.set_ylabel("event")
    ax_psth.legend(fontsize="xx-small", frameon=False, ncol=2)
    ax_raster.set_title(f"{run.name}\nselectivity {row['selectivity']:.1f} (stimuli only {row['selectivity_stim']:.1f}), "
                        f"response {row['response']:+.1f} Hz", fontsize="small")
    rates = sp.condition_rates(times, events["onset"], events["condition"], run.stim_window)
    sp.plot_tuning(ax_tuning, rates, baseline=row["baseline_rate"], color="k")


def plot_sparse_noise_column(fig, cell, run, unit, row):
    inner = cell.subgridspec(3, 1, height_ratios=[1, 1, 1.3], hspace=0.45)
    times, onsets = run.spikes.spikes(unit), run.events["onset"].to_numpy()
    kernel = rf.sparse_noise_kernel(times, onsets, run.frames, rf_lags, bin_size=0.02)
    on, off = rf.sparse_noise_maps(times, onsets, run.frames)
    ax_on, ax_off = fig.add_subplot(inner[0]), fig.add_subplot(inner[1])
    images = rf.plot_maps([ax_on, ax_off], on, off)
    fig.colorbar(images[1], ax=[ax_on, ax_off], label="Δ rate (Hz)", shrink=0.8)
    ax_on.set_title(f"{run.name}: receptive field strength {row['selectivity']:.1f}\nON (white squares)",
                    fontsize="small")
    # Time course at the square (and sign) with the largest response in the mapping window
    sign, r, c = np.unravel_index(np.argmax(np.abs(np.stack([on, off]))), (2, *on.shape))
    for ax in (ax_on, ax_off):
        ax.plot(c, r, "s", mfc="none", mec="k", ms=8)
    ax = fig.add_subplot(inner[2])
    ax.plot(rf_lags + 0.01, kernel[:, 0, r, c], color="C3", label="ON")
    ax.plot(rf_lags + 0.01, kernel[:, 1, r, c], color="C0", label="OFF")
    ax.axhline(0, color="0.5", lw=0.8)
    ax.axvspan(0.03, 0.13, color="0.9", zorder=0)
    ax.set(xlabel="time after square appeared (s)", ylabel="Δ rate (Hz)", title=f"square at row {r}, column {c}")
    ax.legend(fontsize="x-small", frameon=False)


def plot_unit_summary(unit, shown, filter_name, passing):
    """One column per run in ``shown`` for this unit, plus a probe map."""
    fig = plt.figure(figsize=(4.4 * len(shown) + 2.4, 9))
    grid = fig.add_gridspec(1, len(shown) + 1, width_ratios=[4.4] * len(shown) + [2.4], wspace=0.35)
    for i, name in enumerate(shown):
        run, row = runs[name], scores[name].loc[unit]
        (plot_sparse_noise_column if run.kind == "sparse_noise" else plot_events_column)(fig, grid[0, i], run, unit, row)
    units = scores[filter_name]
    ax_map = fig.add_subplot(grid[0, -1])
    ax_map.scatter(units["x"], units["y"], s=8, c="0.85")
    ax_map.scatter(passing["x"], passing["y"], s=10, c="C0")
    ax_map.scatter(units.loc[unit, "x"], units.loc[unit, "y"], s=160, c="C3", marker="*")
    ax_map.set(xlabel="x (µm)", ylabel="distance from tip (µm)", title="probe")
    fig.suptitle(f"unit {unit} ({units.loc[unit, 'label']}), y = {units.loc[unit, 'y']:.0f} µm")
    return fig


recording_w = w.Dropdown(options=names, description="Recording")
label_w = w.ToggleButtons(options=["all", "good", "mua"], description="Units")
min_w = w.FloatLogSlider(value=3, base=10, min=-1, max=3, step=0.05, description="Min selectivity",
                         readout_format=".1f", continuous_update=False, style={"description_width": "initial"})
blank_w = w.Checkbox(value=False, description="Ignore blank (tuning among stimuli)")
sort_w = w.Dropdown(options=["selectivity", "response", "depth", "unit number"], description="Sort by")
all_w = w.Checkbox(value=True, description="Show all recordings")
unit_w = w.Dropdown(description="Unit", layout=w.Layout(width="440px"))
previous_w = w.Button(description="◀ previous", layout=w.Layout(width="110px"))
next_w = w.Button(description="next ▶", layout=w.Layout(width="110px"))
count_w = w.HTML()
out = w.Output()


def passing_units():
    """Units of the chosen recording that pass the filters, in the chosen order."""
    units = scores[recording_w.value]
    column = "selectivity_stim" if blank_w.value else "selectivity"
    keep = units[column] >= min_w.value
    if label_w.value != "all":
        keep &= units["label"] == label_w.value
    units = units[keep]
    if sort_w.value == "unit number":
        return units.sort_index(), column
    by = {"selectivity": column, "response": "response", "depth": "y"}[sort_w.value]
    return units.sort_values(by, ascending=False), column


def draw(*_):
    out.clear_output(wait=True)
    with out:
        if unit_w.value is None:
            print("No units pass the filters: lower the minimum selectivity or change the label.")
            return
        passing, _ = passing_units()
        shown = names if all_w.value else [recording_w.value]
        plot_unit_summary(unit_w.value, shown, recording_w.value, passing)
        plt.show()


def refresh_units(*_):
    passing, column = passing_units()
    current = unit_w.value
    options = [(f"{u}  ({r['label']}, selectivity {r[column]:.1f}, y {r['y']:.0f} µm)", u) for u, r in passing.iterrows()]
    unit_w.unobserve(draw, "value")
    unit_w.options = options
    unit_w.value = current if current in passing.index else (options[0][1] if options else None)
    unit_w.observe(draw, "value")
    count_w.value = f"<b>{len(passing)}</b> of {len(scores[recording_w.value])} units pass"
    draw()


def step(direction):
    values = [value for _, value in unit_w.options]
    if values and unit_w.value in values:
        unit_w.value = values[(values.index(unit_w.value) + direction) % len(values)]


for control in (recording_w, label_w, min_w, blank_w, sort_w):
    control.observe(refresh_units, "value")
all_w.observe(draw, "value")
unit_w.observe(draw, "value")
previous_w.on_click(lambda _: step(-1))
next_w.on_click(lambda _: step(+1))

display(w.VBox([
    w.HBox([recording_w, sort_w, all_w]),
    w.HBox([label_w, blank_w]),
    w.HBox([min_w, count_w]),
    w.HBox([unit_w, previous_w, next_w]),
    out,
]))
refresh_units()

## Table of the units passing the filters

Re-run this cell after changing the controls to list the units (and copy unit numbers into other analyses).

In [ ]:
passing, column = passing_units()
passing.round(2)